# CuPy - GPU 加速的 NumPy 替代方案

> CuPy 提供與 NumPy 幾乎相同的 API，讓你的陣列運算在 GPU 上獲得 100 倍以上的加速。

## GPU 加速系列 Notebooks

1. [Numba CUDA 基礎](./01-Introduction-to-cuda-python-with-numba.ipynb)
2. [GPU 加速概覽與環境建置](./02-GPU-Acceleration-Overview-and-Environment-Setup.ipynb)
3. **[CuPy - GPU 版 NumPy](./03-CuPy-GPU-Accelerated-NumPy.ipynb) ← 目前位置**
4. [cuDF - GPU 版 pandas](./04-cuDF-GPU-Accelerated-DataFrames.ipynb)
5. [cuML - GPU 版 scikit-learn](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb)
6. [cuGraph - GPU 版 NetworkX](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb)
7. [Dask-CUDA 多 GPU 處理](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb)
8. [RAPIDS 端到端實戰](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb)

# 環境檢查 - GPU 可用性
import shutil
import subprocess
import platform

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    if platform.system() == 'Darwin' and platform.machine() == 'arm64':
        print('Apple Silicon 不支援 CUDA/RAPIDS。')
        print('Mac 替代方案: pip install polars (DataFrame), pip install mlx (ML)')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
# 環境檢查 - GPU 可用性
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
# 安裝 CuPy（依 CUDA 版本選擇）
# !pip install cupy-cuda12x    # CUDA 12
# !pip install cupy-cuda11x    # CUDA 11

In [ ]:
import numpy as np
import time

if GPU_AVAILABLE:
    import cupy as cp
    print(f'CuPy 版本: {cp.__version__}')
    print(f'CUDA 版本: {cp.cuda.runtime.runtimeGetVersion()}')
    # 顯示 GPU 資訊
    dev = cp.cuda.Device(0)
    props = cp.cuda.runtime.getDeviceProperties(dev.id)
    print(f'GPU: {props["name"].decode()}')
    mem = dev.mem_info
    print(f'VRAM: {mem[1] / 1e9:.1f} GB (可用: {mem[0] / 1e9:.1f} GB)')
else:
    print('本 Notebook 需要 GPU 環境。CPU 範例仍可參考學習。')

---
## 1. CuPy vs NumPy API 對照

CuPy 的設計目標是 **NumPy 的 drop-in replacement**（直接替代）。大部分情況下只需把 `np` 改成 `cp`：

| NumPy | CuPy | 說明 |
|-------|------|------|
| `np.array([1,2,3])` | `cp.array([1,2,3])` | 建立陣列 |
| `np.zeros((3,3))` | `cp.zeros((3,3))` | 零矩陣 |
| `np.random.rand(N)` | `cp.random.rand(N)` | 隨機數 |
| `np.dot(a, b)` | `cp.dot(a, b)` | 矩陣乘法 |
| `np.sum(a)` | `cp.sum(a)` | 加總 |
| `np.fft.fft(a)` | `cp.fft.fft(a)` | 傅立葉轉換 |
| `np.linalg.inv(a)` | `cp.linalg.inv(a)` | 逆矩陣 |

> **重點：** CuPy 陣列存在 GPU 記憶體 (VRAM) 中，NumPy 陣列存在 CPU 記憶體 (RAM) 中。

---
## 2. 基本操作

In [ ]:
# 建立陣列
if GPU_AVAILABLE:
    # GPU 陣列
    a_gpu = cp.array([1, 2, 3, 4, 5])
    print(f'CuPy 陣列: {a_gpu}')
    print(f'型別: {type(a_gpu)}')
    print(f'儲存位置: GPU (device {a_gpu.device})')
    print()

    # CPU 陣列
    a_cpu = np.array([1, 2, 3, 4, 5])
    print(f'NumPy 陣列: {a_cpu}')
    print(f'型別: {type(a_cpu)}')
    print(f'儲存位置: CPU (RAM)')
else:
    a_cpu = np.array([1, 2, 3, 4, 5])
    print(f'NumPy 陣列: {a_cpu}')
    print('(GPU 不可用，僅示範 NumPy)')

In [ ]:
# CPU ↔ GPU 資料轉換
if GPU_AVAILABLE:
    # NumPy → CuPy (CPU → GPU)
    np_array = np.array([10, 20, 30])
    cp_array = cp.asarray(np_array)     # 傳入 GPU
    print(f'CPU → GPU: {cp_array} (type: {type(cp_array)})')

    # CuPy → NumPy (GPU → CPU)
    back_to_cpu = cp.asnumpy(cp_array)  # 傳回 CPU
    # 或者用 cp_array.get()
    print(f'GPU → CPU: {back_to_cpu} (type: {type(back_to_cpu)})')

---
## 3. 效能比較範例

### 3.1 矩陣乘法 (Matrix Multiplication)

In [ ]:
N = 5000  # 矩陣大小 (可調整：VRAM 不足時降低)

# ----- CPU -----
a_cpu = np.random.rand(N, N).astype(np.float32)
b_cpu = np.random.rand(N, N).astype(np.float32)

start = time.time()
c_cpu = np.dot(a_cpu, b_cpu)
cpu_time = time.time() - start
print(f'CPU 矩陣乘法 ({N}x{N}): {cpu_time:.4f} 秒')

# ----- GPU -----
if GPU_AVAILABLE:
    a_gpu = cp.asarray(a_cpu)
    b_gpu = cp.asarray(b_cpu)

    # 暖機 (warm-up)：第一次執行包含 kernel 編譯時間
    _ = cp.dot(a_gpu, b_gpu)
    cp.cuda.Stream.null.synchronize()

    start = time.time()
    c_gpu = cp.dot(a_gpu, b_gpu)
    cp.cuda.Stream.null.synchronize()  # 確保 GPU 運算完成才停止計時
    gpu_time = time.time() - start

    print(f'GPU 矩陣乘法 ({N}x{N}): {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

### 3.2 元素級運算 (Element-wise Operations)

In [ ]:
N = 50_000_000  # 五千萬個元素

# CPU
arr_cpu = np.random.rand(N).astype(np.float32)

start = time.time()
result_cpu = np.sin(arr_cpu) ** 2 + np.cos(arr_cpu) ** 2  # 應該 ≈ 1
cpu_time = time.time() - start
print(f'CPU 元素級運算 ({N:,} 元素): {cpu_time:.4f} 秒')

# GPU
if GPU_AVAILABLE:
    arr_gpu = cp.asarray(arr_cpu)

    # 暖機
    _ = cp.sin(arr_gpu) ** 2 + cp.cos(arr_gpu) ** 2
    cp.cuda.Stream.null.synchronize()

    start = time.time()
    result_gpu = cp.sin(arr_gpu) ** 2 + cp.cos(arr_gpu) ** 2
    cp.cuda.Stream.null.synchronize()
    gpu_time = time.time() - start

    print(f'GPU 元素級運算 ({N:,} 元素): {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

### 3.3 傅立葉轉換 (FFT)

In [ ]:
N = 10_000_000  # 一千萬個資料點

# 模擬訊號：多個正弦波疊加
t_cpu = np.linspace(0, 1, N, dtype=np.float32)
signal_cpu = (np.sin(2 * np.pi * 50 * t_cpu) +
              0.5 * np.sin(2 * np.pi * 120 * t_cpu) +
              0.3 * np.random.randn(N).astype(np.float32))  # 加入雜訊

# CPU FFT
start = time.time()
fft_cpu = np.fft.fft(signal_cpu)
cpu_time = time.time() - start
print(f'CPU FFT ({N:,} 點): {cpu_time:.4f} 秒')

# GPU FFT
if GPU_AVAILABLE:
    signal_gpu = cp.asarray(signal_cpu)

    _ = cp.fft.fft(signal_gpu)
    cp.cuda.Stream.null.synchronize()

    start = time.time()
    fft_gpu = cp.fft.fft(signal_gpu)
    cp.cuda.Stream.null.synchronize()
    gpu_time = time.time() - start

    print(f'GPU FFT ({N:,} 點): {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

### 3.4  2D 卷積 (Convolution)

In [ ]:
from scipy import signal as sp_signal

# 模擬大型影像
image_size = 4000
kernel_size = 15

image_cpu = np.random.rand(image_size, image_size).astype(np.float32)
kernel_cpu = np.random.rand(kernel_size, kernel_size).astype(np.float32)

# CPU 2D 卷積
start = time.time()
result_cpu = sp_signal.fftconvolve(image_cpu, kernel_cpu, mode='same')
cpu_time = time.time() - start
print(f'CPU 2D 卷積 ({image_size}x{image_size}, kernel {kernel_size}x{kernel_size}): {cpu_time:.4f} 秒')

# GPU 2D 卷積
if GPU_AVAILABLE:
    from cupyx.scipy import signal as cp_signal

    image_gpu = cp.asarray(image_cpu)
    kernel_gpu = cp.asarray(kernel_cpu)

    # 暖機
    _ = cp_signal.fftconvolve(image_gpu, kernel_gpu, mode='same')
    cp.cuda.Stream.null.synchronize()

    start = time.time()
    result_gpu = cp_signal.fftconvolve(image_gpu, kernel_gpu, mode='same')
    cp.cuda.Stream.null.synchronize()
    gpu_time = time.time() - start

    print(f'GPU 2D 卷積 ({image_size}x{image_size}, kernel {kernel_size}x{kernel_size}): {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

---
## 4. GPU 記憶體管理

GPU 記憶體 (VRAM) 是有限的資源，需要注意管理。

In [ ]:
if GPU_AVAILABLE:
    # 查看目前 GPU 記憶體使用量
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()

    print('GPU 記憶體狀態：')
    print(f'  已使用:   {mempool.used_bytes() / 1e6:.1f} MB')
    print(f'  已分配:   {mempool.total_bytes() / 1e6:.1f} MB')
    print()

    # 建立大陣列觀察記憶體變化
    big_array = cp.random.rand(10000, 10000, dtype=cp.float32)  # ~400 MB
    print(f'建立 10000x10000 陣列後:')
    print(f'  已使用:   {mempool.used_bytes() / 1e6:.1f} MB')

    # 釋放記憶體
    del big_array
    mempool.free_all_blocks()  # 強制釋放快取
    print(f'\n釋放後:')
    print(f'  已使用:   {mempool.used_bytes() / 1e6:.1f} MB')

### 記憶體管理要點

- `cp.get_default_memory_pool()` — 查看記憶體池狀態
- `del array` + `mempool.free_all_blocks()` — 釋放不需要的 GPU 記憶體
- CuPy 預設使用 **記憶體池 (memory pool)**，不會立即釋放已刪除陣列的記憶體
- 如果遇到 `OutOfMemoryError`，先嘗試釋放不需要的變數

---
## 5. Host-Device 傳輸成本

頻繁的 CPU ↔ GPU 傳輸會抵消加速效果。讓我們量化這個成本：

In [ ]:
if GPU_AVAILABLE:
    sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

    print(f'{"資料量":>12s}  {"CPU→GPU":>10s}  {"GPU→CPU":>10s}  {"GPU 運算":>10s}')
    print('-' * 50)

    for n in sizes:
        data_cpu = np.random.rand(n).astype(np.float32)

        # CPU → GPU 傳輸時間
        start = time.time()
        data_gpu = cp.asarray(data_cpu)
        cp.cuda.Stream.null.synchronize()
        h2d_time = time.time() - start

        # GPU 運算時間
        _ = cp.sum(data_gpu)  # 暖機
        cp.cuda.Stream.null.synchronize()
        start = time.time()
        _ = cp.sin(data_gpu) * cp.cos(data_gpu) + cp.sqrt(cp.abs(data_gpu))
        cp.cuda.Stream.null.synchronize()
        compute_time = time.time() - start

        # GPU → CPU 傳輸時間
        start = time.time()
        _ = cp.asnumpy(data_gpu)
        d2h_time = time.time() - start

        print(f'{n:>12,d}  {h2d_time*1000:>8.2f}ms  {d2h_time*1000:>8.2f}ms  {compute_time*1000:>8.2f}ms')

    print()
    print('觀察：資料量小時，傳輸成本 > 運算加速 → 不值得用 GPU')

---
## 6. 反模式：何時不該用 CuPy

### 反模式 1：小陣列

In [ ]:
# 小陣列：CPU 通常更快
small_n = 100

a_cpu = np.random.rand(small_n, small_n).astype(np.float32)

# CPU
start = time.time()
for _ in range(1000):
    _ = np.dot(a_cpu, a_cpu)
cpu_time = time.time() - start
print(f'CPU 1000 次 {small_n}x{small_n} 矩陣乘法: {cpu_time:.4f} 秒')

# GPU
if GPU_AVAILABLE:
    a_gpu = cp.asarray(a_cpu)
    _ = cp.dot(a_gpu, a_gpu)
    cp.cuda.Stream.null.synchronize()

    start = time.time()
    for _ in range(1000):
        _ = cp.dot(a_gpu, a_gpu)
    cp.cuda.Stream.null.synchronize()
    gpu_time = time.time() - start

    print(f'GPU 1000 次 {small_n}x{small_n} 矩陣乘法: {gpu_time:.4f} 秒')
    ratio = cpu_time / gpu_time
    faster = 'GPU' if ratio > 1 else 'CPU'
    print(f'→ {faster} 較快 ({max(ratio, 1/ratio):.1f}x)')
    print()
    print('結論：小矩陣的 kernel launch overhead 抵消了 GPU 的平行優勢')

### 反模式 2：頻繁 CPU-GPU 傳輸

In [ ]:
if GPU_AVAILABLE:
    N = 1_000_000

    # 不好的做法：每次迭代都傳輸
    data = np.random.rand(N).astype(np.float32)
    start = time.time()
    for _ in range(10):
        gpu_data = cp.asarray(data)      # CPU → GPU
        result = cp.sum(gpu_data ** 2)
        data = cp.asnumpy(gpu_data + 1)  # GPU → CPU
    cp.cuda.Stream.null.synchronize()
    bad_time = time.time() - start
    print(f'頻繁傳輸 (10 次來回): {bad_time:.4f} 秒')

    # 好的做法：一次傳入，全程 GPU
    data = np.random.rand(N).astype(np.float32)
    start = time.time()
    gpu_data = cp.asarray(data)           # CPU → GPU（只傳一次）
    for _ in range(10):
        result = cp.sum(gpu_data ** 2)
        gpu_data = gpu_data + 1            # 全程在 GPU
    final = cp.asnumpy(gpu_data)          # GPU → CPU（只傳一次）
    cp.cuda.Stream.null.synchronize()
    good_time = time.time() - start
    print(f'最少傳輸 (1 次來回):  {good_time:.4f} 秒')
    print(f'差異: {bad_time / good_time:.1f}x')

---
## 7. 練習：Monte Carlo 圓周率估算

經典的 Monte Carlo 模擬：隨機撒點估算 π 值。

原理：在 1x1 正方形內隨機撒 N 個點，計算落在 1/4 圓內的比例 → π ≈ 4 × (圓內點數 / 總點數)

In [ ]:
def monte_carlo_pi_cpu(n_samples):
    """CPU 版本的 Monte Carlo π 估算"""
    x = np.random.rand(n_samples).astype(np.float32)
    y = np.random.rand(n_samples).astype(np.float32)
    inside = np.sum(x**2 + y**2 <= 1.0)
    return 4.0 * inside / n_samples

# CPU 版本
N = 100_000_000  # 一億個點

start = time.time()
pi_cpu = monte_carlo_pi_cpu(N)
cpu_time = time.time() - start
print(f'CPU Monte Carlo ({N:,} 點)')
print(f'  π ≈ {pi_cpu:.6f} (誤差: {abs(pi_cpu - np.pi):.6f})')
print(f'  耗時: {cpu_time:.4f} 秒')

In [ ]:
# GPU 版本
if GPU_AVAILABLE:
    def monte_carlo_pi_gpu(n_samples):
        """GPU 版本的 Monte Carlo π 估算"""
        x = cp.random.rand(n_samples, dtype=cp.float32)
        y = cp.random.rand(n_samples, dtype=cp.float32)
        inside = cp.sum(x**2 + y**2 <= 1.0)
        return 4.0 * float(inside) / n_samples

    # 暖機
    _ = monte_carlo_pi_gpu(1000)
    cp.cuda.Stream.null.synchronize()

    start = time.time()
    pi_gpu = monte_carlo_pi_gpu(N)
    cp.cuda.Stream.null.synchronize()
    gpu_time = time.time() - start

    print(f'GPU Monte Carlo ({N:,} 點)')
    print(f'  π ≈ {pi_gpu:.6f} (誤差: {abs(pi_gpu - np.pi):.6f})')
    print(f'  耗時: {gpu_time:.4f} 秒')
    print(f'  加速: {cpu_time / gpu_time:.1f}x')
else:
    print('需要 GPU 環境執行。預期加速: 30-100x')

---
## 8. 效能總結

| 運算類型 | 典型加速倍數 | 適合場景 |
|----------|-------------|----------|
| 矩陣乘法 | 50-100x | 大矩陣 (>1000x1000) |
| 元素級運算 | 30-80x | 大陣列 (>100 萬元素) |
| FFT | 20-50x | 長訊號 (>100 萬點) |
| 2D 卷積 | 30-80x | 大影像 (>2000x2000) |
| Monte Carlo | 30-100x | 大量隨機模擬 |
| 小陣列運算 | **0.1-1x (更慢)** | **不適合 GPU** |

### 經驗法則

- 陣列元素 > **100 萬** → 考慮用 CuPy
- 矩陣維度 > **1000** → 幾乎一定值得用 GPU
- 需要多次迭代的運算 → 把資料留在 GPU 上
- 一次性小運算 → 用 NumPy 就好

---
## 參考資源

- [CuPy 官方文件](https://docs.cupy.dev/en/stable/)
- [CuPy vs NumPy 比較](https://docs.cupy.dev/en/stable/user_guide/difference.html)
- [CuPy GitHub](https://github.com/cupy/cupy)

> **下一步：** 前往 [04-cuDF-GPU-Accelerated-DataFrames.ipynb](./04-cuDF-GPU-Accelerated-DataFrames.ipynb) 學習 GPU 加速的 DataFrame 操作。